# Aligning Coronavirus Spike Proteins with Progressive Alignment and BLOSUM62

In [1]:
with open("../data/BLOSUM62.txt") as f:
    for line in f:
        print(line)

#  BLOSUM62 substitution matrix (Henikoff & Henikoff, 1992)

#  Exported from Biopython (Bio.Align.substitution_matrices)

     A  R  N  D  C  Q  E  G  H  I  L  K  M  F  P  S  T  W  Y  V  B  Z  X  *

A    4 -1 -2 -2  0 -1 -1  0 -2 -1 -1 -1 -1 -2 -1  1  0 -3 -2  0 -2 -1  0 -4

R   -1  5  0 -2 -3  1  0 -2  0 -3 -2  2 -1 -3 -2 -1 -1 -3 -2 -3 -1  0 -1 -4

N   -2  0  6  1 -3  0  0  0  1 -3 -3  0 -2 -3 -2  1  0 -4 -2 -3  3  0 -1 -4

D   -2 -2  1  6 -3  0  2 -1 -1 -3 -4 -1 -3 -3 -1  0 -1 -4 -3 -3  4  1 -1 -4

C    0 -3 -3 -3  9 -3 -4 -3 -3 -1 -1 -3 -1 -2 -3 -1 -1 -2 -2 -1 -3 -3 -2 -4

Q   -1  1  0  0 -3  5  2 -2  0 -3 -2  1  0 -3 -1  0 -1 -2 -1 -2  0  3 -1 -4

E   -1  0  0  2 -4  2  5 -2  0 -3 -3  1 -2 -3 -1  0 -1 -3 -2 -2  1  4 -1 -4

G    0 -2  0 -1 -3 -2 -2  6 -2 -4 -4 -2 -3 -3 -2  0 -2 -2 -3 -3 -1 -2 -1 -4

H   -2  0  1 -1 -3  0  0 -2  8 -3 -3 -1 -2 -1 -2 -1 -2 -2  2 -3  0  0 -1 -4

I   -1 -3 -3 -3 -1 -3 -3 -4 -3  4  2 -3  1  0 -3 -2 -1 -3 -1  3 -3 -3 -1 -4

L   -1 -2 -3 -4 -1 -2 -3 -4 -3

In [2]:
line = "A    4 -1 -2  0"
parts = line.split()
print(parts)
print(parts[0])
print(parts[1:])

['A', '4', '-1', '-2', '0']
A
['4', '-1', '-2', '0']


In [3]:
def load_blosum62(file_path):
    """Read a BLOSUM62 file and return a dictionary of scores.

    The keys are pairs of letters, for example ("A", "R"),
    and the values are whole-number scores, for example -1.
    """
    score = {}
    column_letters = None

    with open(file_path) as f:
        for line in f:
            line = line.strip()                     # remove spaces and newline

            if line == "":                          # skip blank lines
                continue
            if line.startswith("#"):                # skip comments
                continue

            if column_letters is None:              # first real line = header
                column_letters = line.split()
                continue

            parts = line.split()                    # every other line = a row
            row_letter = parts[0]
            values = parts[1:]

            for column_letter, value in zip(column_letters, values):
                score[(row_letter, column_letter)] = int(value)

    return score

In [4]:
score = load_blosum62("../data/BLOSUM62.txt")
print(score[("E", "A")])
print(score[("W", "W")])
print(score[("H", "H")])
print(len(score))

-1
11
8
576


In [5]:
def check_sequence(sequence, name, score):
    """Clean a protein sequence and check every letter is in BLOSUM62.

    Returns the cleaned sequence. Stops with an error message if the
    sequence is empty or contains a letter that has no BLOSUM62 score.
    """
    sequence = sequence.upper()                  # "heag" -> "HEAG"
    sequence = "".join(sequence.split())         # remove all spaces and line breaks

    if sequence == "":
        raise ValueError(f"Sequence {name} is empty.")

    allowed_letters = {row_letter for (row_letter, column_letter) in score}

    for position, letter in enumerate(sequence, start=1):
        if letter not in allowed_letters:
            raise ValueError(
                f"{name} has an unexpected letter {letter} at position {position}."
            )

    return sequence

In [6]:
print(check_sequence("heag", "test1", score))          # lowercase is fixed
print(check_sequence("HE AG\nKL", "test2", score))     # spaces and line breaks removed

for seq, name in [("", "empty_test"), ("HEJAG", "bad_letter_test")]:
    try:
        check_sequence(seq, name, score)
    except ValueError as error:
        print("Stopped:", error)

HEAG
HEAGKL
Stopped: Sequence empty_test is empty.
Stopped: bad_letter_test has an unexpected letter J at position 3.


In [7]:
def needleman_wunsch(x, y, score, gap_penalty, show_grid=False):
    """Globally align x (across the top) and y (down the side).

    Returns the two aligned sequences and the alignment score.
    If show_grid is True, also prints the score grid and trace grid.
    """
    rows = len(y) + 1                      # one extra row for the "start" edge
    cols = len(x) + 1                      # one extra column for the "start" edge

    # Part 1: set up two grids
    grid = [[0] * cols for _ in range(rows)]      # the scores
    trace = [[""] * cols for _ in range(rows)]    # the arrows: "D", "U" or "L"

    # Part 2: fill the edges
    for j in range(1, cols):                       # top row
        grid[0][j] = grid[0][j - 1] + gap_penalty
        trace[0][j] = "L"
    for i in range(1, rows):                       # left column
        grid[i][0] = grid[i - 1][0] + gap_penalty
        trace[i][0] = "U"

    # Part 3: fill the rest, row by row, left to right
    for i in range(1, rows):
        for j in range(1, cols):
            diagonal = grid[i - 1][j - 1] + score[(y[i - 1], x[j - 1])]
            up = grid[i - 1][j] + gap_penalty
            left = grid[i][j - 1] + gap_penalty

            best = max(diagonal, up, left)
            grid[i][j] = best

            # ties: prefer diagonal, then up, then left
            if best == diagonal:
                trace[i][j] = "D"
            elif best == up:
                trace[i][j] = "U"
            else:
                trace[i][j] = "L"

    if show_grid:
        print("Score grid:")
        for row in grid:
            print(row)
        print("Trace grid:")
        for row in trace:
            print(row)

    # Part 4: trace back from the bottom-right
    aligned_x = []
    aligned_y = []
    i = rows - 1
    j = cols - 1

    while i > 0 or j > 0:
        move = trace[i][j]
        if move == "D":                    # pair the two letters
            aligned_x.append(x[j - 1])
            aligned_y.append(y[i - 1])
            i -= 1
            j -= 1
        elif move == "U":                  # gap in x
            aligned_x.append("-")
            aligned_y.append(y[i - 1])
            i -= 1
        else:                              # "L": gap in y
            aligned_x.append(x[j - 1])
            aligned_y.append("-")
            j -= 1

    aligned_x.reverse()                    # built backwards, so flip
    aligned_y.reverse()

    return "".join(aligned_x), "".join(aligned_y), grid[rows - 1][cols - 1]

In [8]:
#TEST
aligned_x, aligned_y, alignment_score = needleman_wunsch("HEAG", "HAG", score, -10, show_grid=True)
print(aligned_x)
print(aligned_y)
print("Score:", alignment_score)

Score grid:
[0, -10, -20, -30, -40]
[-10, 8, -2, -12, -22]
[-20, -2, 7, 2, -8]
[-30, -12, -3, 7, 8]
Trace grid:
['', 'L', 'L', 'L', 'L']
['U', 'D', 'L', 'L', 'L']
['U', 'U', 'D', 'D', 'L']
['U', 'U', 'U', 'D', 'D']
HEAG
H-AG
Score: 8


In [9]:
from Bio import Align
from Bio.Align import substitution_matrices

aligner = Align.PairwiseAligner(mode="global")
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
aligner.open_gap_score = -10
aligner.extend_gap_score = -10

print("Biopython:", aligner.score("HEAG", "HAG"))
print("Mine:     ", needleman_wunsch("HEAG", "HAG", score, -10)[2])

Biopython: 8.0
Mine:      8
